In [127]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings as wr
wr.filterwarnings('ignore')

In [128]:
df = pd.read_csv('07_flats_EDA.csv')
df.dropna(inplace=True)    # Dropping rows with missing values for simplicity (can be handled better with imputation if needed)
df['price_cr'] = df['price_cr']*1.7      # Adjusting price to current market value (approx. 70% increase from 2021 to 2026)

In [129]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [130]:
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

In [131]:
ct = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oe', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_features),
], remainder='passthrough')

X = ct.fit_transform(X)

In [132]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [133]:
def evaluate_model(y, y_pred):
    mae = mean_absolute_error(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, y_pred)
    return mae, rmse, r2

In [154]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'KNN Regressor': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'Ada Boost': AdaBoostRegressor(),
    'SVM': SVR(kernel='rbf', C=70, gamma=0.06, epsilon=.14),
    'XGBoost': XGBRegressor(),
    'CatBoost': CatBoostRegressor(verbose=False)
}

In [155]:
test_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    test_scores[name] = test_r2

    print(f"{name}:")
    print(f"Training - MAE: {train_mae:.2f}, RMSE: {train_rmse:.2f}, R²: {train_r2:.4f}")
    print(f"Testing  - MAE: {test_mae:.2f}, RMSE: {test_rmse:.2f}, R²: {test_r2:.4f}\n")

best_model_name = max(test_scores, key=test_scores.get)
print(f"Best Model: {best_model_name} with R²: {test_scores[best_model_name]:.4f}")

Linear Regression:
Training - MAE: 0.93, RMSE: 1.51, R²: 0.6126
Testing  - MAE: 0.91, RMSE: 1.63, R²: 0.4664

Ridge Regression:
Training - MAE: 0.93, RMSE: 1.51, R²: 0.6126
Testing  - MAE: 0.91, RMSE: 1.63, R²: 0.4668

Lasso Regression:
Training - MAE: 1.28, RMSE: 2.03, R²: 0.2996
Testing  - MAE: 1.22, RMSE: 1.83, R²: 0.3211

KNN Regressor:
Training - MAE: 0.49, RMSE: 1.04, R²: 0.8150
Testing  - MAE: 0.62, RMSE: 1.25, R²: 0.6828

Decision Tree:
Training - MAE: 0.02, RMSE: 0.07, R²: 0.9992
Testing  - MAE: 0.47, RMSE: 1.09, R²: 0.7586

Random Forest:
Training - MAE: 0.16, RMSE: 0.34, R²: 0.9801
Testing  - MAE: 0.38, RMSE: 0.78, R²: 0.8773

Gradient Boosting:
Training - MAE: 0.43, RMSE: 0.67, R²: 0.9236
Testing  - MAE: 0.53, RMSE: 0.91, R²: 0.8313

Ada Boost:
Training - MAE: 1.36, RMSE: 1.58, R²: 0.5744
Testing  - MAE: 1.41, RMSE: 1.70, R²: 0.4167

SVM:
Training - MAE: 0.12, RMSE: 0.14, R²: 0.9965
Testing  - MAE: 0.53, RMSE: 1.11, R²: 0.7511

XGBoost:
Training - MAE: 0.10, RMSE: 0.15, R²: